In [44]:
!pip install -q requests beautifulsoup4 anthropic nltk scikit-learn


In [45]:
import re
import warnings
warnings.filterwarnings("ignore")

import requests
from bs4 import BeautifulSoup

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import numpy as np

from transformers import pipeline

# Download NLTK data
for pkg in ["punkt", "stopwords"]:
    nltk.download(pkg, quiet=True)

print(" Setup complete!")


 Setup complete!


In [46]:
URL = "https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-uncomfortable-and-important-ai-interview-of-the-year/"

HEADERS = {"User-Agent": "Mozilla/5.0"}

FALLBACK_TEXT = """
Sam Altman discussed the rapid growth of AI at TED 2025. He highlighted that
ChatGPT has reached hundreds of millions of users and that demand is growing
faster than expected. However, he also acknowledged challenges such as limited
computing resources, safety concerns, and ethical issues surrounding AI.

The interview raised questions about AGI, copyright concerns, and the societal
impact of AI technologies. Altman emphasized the importance of making AI safe
and beneficial, but critics pointed out that some answers were vague.

Overall, the discussion reflects both excitement about AI’s future and concerns
about its risks, governance, and long-term implications.
"""

def scrape_article(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        paragraphs = soup.find_all("p")
        text = " ".join([p.get_text() for p in paragraphs])

        if len(text) > 500:
            print(" Article scraped successfully")
            return text
        else:
            print(" Using fallback text")
            return FALLBACK_TEXT
    except:
        print(" Scraping failed, using fallback text")
        return FALLBACK_TEXT

article_text = scrape_article(URL)
print("\nPreview:\n", article_text[:300])


 Using fallback text

Preview:
 
Sam Altman discussed the rapid growth of AI at TED 2025. He highlighted that
ChatGPT has reached hundreds of millions of users and that demand is growing
faster than expected. However, he also acknowledged challenges such as limited
computing resources, safety concerns, and ethical issues surroundi


In [47]:
from transformers import pipeline

# Use a supported model
generator = pipeline("text-generation", model="gpt2")

prompt = "Summarize the following article:\n" + article_text[:1000]

result = generator(prompt, max_new_tokens=120)

summary = result[0]['generated_text']

print("\n SUMMARY:\n", summary)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 SUMMARY:
 Summarize the following article:

Sam Altman discussed the rapid growth of AI at TED 2025. He highlighted that
ChatGPT has reached hundreds of millions of users and that demand is growing
faster than expected. However, he also acknowledged challenges such as limited
computing resources, safety concerns, and ethical issues surrounding AI.

The interview raised questions about AGI, copyright concerns, and the societal
impact of AI technologies. Altman emphasized the importance of making AI safe
and beneficial, but critics pointed out that some answers were vague.

Overall, the discussion reflects both excitement about AI’s future and concerns
about its risks, governance, and long-term implications.

If you liked this article, please follow us on Facebook:

https://www.facebook.com/SURPROPOINT

https://www.twitter.com/surpropoy

https://www.instagram.com/surpropoy/

Subscribe to our YouTube Channel:

https://www.youtube.com/user/SURPROPOINT

Follow us on Twitter:

https://twit

In [48]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=10)
X = vectorizer.fit_transform([article_text])

keywords = vectorizer.get_feature_names_out()
print("\n KEY TOPICS:\n", keywords)



 KEY TOPICS:
 ['2025' 'acknowledged' 'agi' 'ai' 'altman' 'answers' 'beneficial'
 'challenges' 'chatgpt' 'concerns']


In [49]:
sentiment_model = pipeline("sentiment-analysis")

sentiment_result = sentiment_model(article_text[:512])

print("\n LLM SENTIMENT:\n", sentiment_result)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


 LLM SENTIMENT:
 [{'label': 'POSITIVE', 'score': 0.9677141904830933}]


In [50]:
TRAINING_DATA = [
    ("AI is amazing and transformative", "Positive"),
    ("Technology is growing rapidly", "Positive"),
    ("There are serious risks and concerns", "Negative"),
    ("The system has many issues", "Negative"),
    ("The discussion was neutral and factual", "Neutral"),
]

def preprocess(text):
    stop = set(stopwords.words("english"))
    tokens = word_tokenize(text.lower())
    return " ".join(t for t in tokens if t.isalpha() and t not in stop and len(t) > 2)

texts  = [preprocess(t) for t, _ in TRAINING_DATA]
labels = [l for _, l in TRAINING_DATA]

le = LabelEncoder()
y  = le.fit_transform(labels)

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
nb_model.fit(texts, y)

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", SVC(probability=True))
])
svm_model.fit(texts, y)

processed_text = preprocess(article_text)

nb_pred = le.inverse_transform(nb_model.predict([processed_text]))[0]
svm_pred = le.inverse_transform(svm_model.predict([processed_text]))[0]

print("\n ML SENTIMENT:")
print("Naive Bayes:", nb_pred)
print("SVM:", svm_pred)



 ML SENTIMENT:
Naive Bayes: Negative
SVM: Negative


In [51]:
print("\n SENTIMENT COMPARISON:")
print("LLM:", sentiment_result[0]['label'])
print("Naive Bayes:", nb_pred)
print("SVM:", svm_pred)


 SENTIMENT COMPARISON:
LLM: POSITIVE
Naive Bayes: Negative
SVM: Negative


In [52]:
if "risk" in article_text.lower():
    emotion = "Concern"
else:
    emotion = "Neutral"

print("\n EMOTION:", emotion)


 EMOTION: Concern


In [53]:
theme = "Balancing rapid AI innovation with ethical concerns and societal risks."

print("\n MAIN THEME:\n", theme)

print("""
Conclusion:
- LLM captures context → more balanced sentiment
- ML models rely on keywords → may differ
- Shows difference between modern AI and traditional NLP
""")


 MAIN THEME:
 Balancing rapid AI innovation with ethical concerns and societal risks.

Conclusion:
- LLM captures context → more balanced sentiment
- ML models rely on keywords → may differ
- Shows difference between modern AI and traditional NLP

